# 🚀 SVOMPTR-9B ELITE TRAINING PIPELINE (Unsloth + QLoRA)

ဒီ Notebook ကို အမှားအယွင်းမရှိ (Bulletproof) ဖြစ်အောင် အောက်ပါ Feature များ ထည့်သွင်းထားပါတယ် -
- **CUDA Linkage Guard**: BitsAndBytes CUDA error များကို အလိုအလျောက် ရှင်းလင်းပေးခြင်း။
- **Smart Memory Management**: T4 GPU (16GB) ပေါ်မှာ MoE model ကို အေးဆေး train နိုင်အောင် VRAM paging စနစ်သုံးထားခြင်း။
- **Universal Prompting**: ChatML format ကို standard အဖြစ် သတ်မှတ်ထားခြင်း။

## 🛠️ 1. Neural Interface & Hardware Setup

In [ ]:
import torch, os, sys, subprocess

def verify_hardware():
    if not torch.cuda.is_available():
        print("❌ FATAL ERROR: No GPU found!")
        print("💡 HOW TO FIX THIS IN GOOGLE COLAB:")
        print("   1. Click on 'Runtime' in the top menu.")
        print("   2. Click on 'Change runtime type'.")
        print("   3. Under 'Hardware accelerator', select 'T4 GPU'.")
        print("   4. Click 'Save' and connect.")
        return False
    
    # Standard device selection (0)
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"✅ Hardware Active: {gpu_name} ({vram:.1f}GB VRAM)")
    return True

if not verify_hardware():
    raise RuntimeError("FATAL ERROR: GPU required for training. Please check Runtime -> Change runtime type.")

print("📦 Synchronizing Neural Core (Fast Path)...")
try:
    import unsloth
    print("✅ Unsloth already present.")
except ImportError:
    print("📥 Installing Optimized GPU Kernels (Estimated: 2-3 mins)...")
    get_ipython().system('pip uninstall unsloth unsloth-zoo xformers -y')
    get_ipython().system('pip install --upgrade --no-cache-dir unsloth unsloth-zoo')
    get_ipython().system('pip install --quiet unsloth trl peft accelerate bitsandbytes')
    get_ipython().system('pip install --quiet datasets tqdm sentencepiece')
    print("💡 TIP: If you see 'ModuleNotFoundError' later, please 'Restart Session' once.")
print("✅ Core Sync Complete.")

os.environ["BITSANDBYTES_NOWELCOME"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1" # Faster model downloads

In [ ]:
from IPython.display import Javascript
def auto_connect():
    display(Javascript('''
    function KeepAlive(){ 
        const btn = document.querySelector("colab-connect-button");
        if (btn) { 
            console.log("♻️ Connection Active");
            btn.click();
        }
    }
    setInterval(KeepAlive, 60000);
    '''))
auto_connect()
print("🛡️ Anti-Idle Guardian Active.")

## 📁 2. Storage Mapping (Google Drive)

In [ ]:
from google.colab import drive
import os

try:
    drive.mount('/content/drive', force_remount=True)
    BASE_DIR = '/content/drive/MyDrive/svomptr_auto_train'
except:
    print("⚠️ Drive mount failed. Local storage will be wiped after session ends.")
    BASE_DIR = '/content/svomptr_auto_train'

PATHS = {
    "checkpoints": os.path.join(BASE_DIR, 'checkpoints'),
    "weights": os.path.join(BASE_DIR, 'final_lora_weights'),
    "datasets": os.path.join(BASE_DIR, 'datasets')
}

for p in PATHS.values(): os.makedirs(p, exist_ok=True)
print(f"✅ Workspace Mapping: {BASE_DIR}")

## 🧪 3. Neural Architecture Loading

In [ ]:
from unsloth import FastLanguageModel
import torch, gc

MAX_SEQ_LENGTH = 2048
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

def clear_memory():
    gc.collect()
    torch.cuda.empty_cache()
clear_memory()

print(f"🛠️ Loading Neural Engine: {MODEL_NAME}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
    trust_remote_code = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("✅ Experts Layered and Ready.")

## 📑 4. Semantic Dataset Formatting

In [ ]:
from datasets import load_dataset
import json, os

DATA_FILE = os.path.join(PATHS["datasets"], 'training_data.jsonl')

if not os.path.exists(DATA_FILE) or os.path.getsize(DATA_FILE) == 0:
    print("📝 Generating default training matrix...")
    seed_data = [{"en": "The teacher is explaining the lesson.", "my": "ဆရာက သင်ခန်းစာကို ရှင်းပြနေတယ်။", "svomptr_structure": "S(The teacher)-V(explaining)-O(the lesson)"}]
    with open(DATA_FILE, 'w', encoding='utf-8') as f:
        for s in seed_data: f.write(json.dumps(s, ensure_ascii=False) + '\n')

dataset = load_dataset("json", data_files=DATA_FILE, split="train")

def apply_chatml_template(examples):
    chats = []
    ens = examples.get("en", [])
    mys = examples.get("my", [])
    strs = examples.get("svomptr_structure", examples.get("target", [""] * len(ens)))
    
    for en, my, s in zip(ens, mys, strs):
        prompt = f"<|im_start|>system\nEnglish-to-Myanmar SVOMPTR Transformer Expert.\n<|im_end|>\n<|im_start|>user\nTranslate: {en}\n<|im_end|>\n<|im_start|>assistant\nTranslation: {my}\nStructure: {s}\n<|im_end|>"
        chats.append(prompt)
    return { "text" : chats }

dataset = dataset.map(apply_chatml_template, batched = True)
print(f"✅ Tokenization Map Created: {len(dataset)} nodes.")

## 🏋️‍♂️ 5. Neural Training Loop (Production Mode)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
import os, torch

if 'model' not in globals() or model is None:
    print("❌ CRITICAL ERROR: Model handle lost. Re-run Cell 3.")
else:
    training_args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 16,
        warmup_steps = 10,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "paged_adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = PATHS["checkpoints"],
        save_steps = 50,
        save_total_limit = 2,
        report_to = "none",
    )

    trainer_kwargs = {
        "model": model,
        "train_dataset": dataset,
        "dataset_text_field": "text",
        "max_seq_length": MAX_SEQ_LENGTH,
        "dataset_num_proc": 2,
        "args": training_args,
    }

    print("🧩 Configuring High-Resilience Trainer...")
    try:
        # Multi-Version Argument Mapping
        trainer = SFTTrainer(**trainer_kwargs, processing_class=tokenizer)
    except TypeError:
        try:
            trainer = SFTTrainer(**trainer_kwargs, tokenizer=tokenizer)
        except TypeError:
            print("⚠️ Fallback: Manual interface assignment.")
            trainer = SFTTrainer(**trainer_kwargs)
            trainer.tokenizer = tokenizer

    print("▶️ Initiating Training Pipeline...")
    try:
        ckpt_dir = PATHS["checkpoints"]
        checkpoints = [d for d in os.listdir(ckpt_dir) if d.startswith("checkpoint-")] if os.path.exists(ckpt_dir) else []
        checkpoints = sorted(checkpoints, key=lambda x: int(x.split("-")[-1])) if checkpoints else []
        latest_ckpt = checkpoints[-1] if checkpoints else None
        resume_path = os.path.join(ckpt_dir, latest_ckpt) if latest_ckpt else None
        
        if resume_path: print(f"📥 Resuming from: {latest_ckpt}")
        else: print("📥 Fresh Start Tracking Engine Active.")
        
        torch.cuda.empty_cache()
        trainer.train(resume_from_checkpoint = resume_path)
        print("✅ Training cycle finished successfully.")
    except Exception as e:
        print(f"❌ Execution Error: {e}")

## 📦 6. Model Solidification

In [ ]:
if model:
    print(f"💾 Storing Neural Weights at {PATHS['weights']}...")
    model.save_pretrained(PATHS['weights'])
    tokenizer.save_pretrained(PATHS['weights'])
    print("✨ Saved! Ready for deployment in the SVOMPTR Brain.")
else:
    print("❌ Cleanup error: No weights to save.")

## 💬 7. Interactive Chat Module

In [ ]:
if 'model' in globals() and model:
    from unsloth import FastLanguageModel
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    print("🔌 Enabling Fast Inference Mode...")
    FastLanguageModel.for_inference(model)
    
    def generate_reply(english_text):
        prompt = f"<|im_start|>system\nEnglish-to-Myanmar SVOMPTR Transformer Expert.\n<|im_end|>\n<|im_start|>user\nTranslate: {english_text}\n<|im_end|>\n<|im_start|>assistant\n"
        inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
        outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True, pad_token_id=tokenizer.eos_token_id)
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
        if "assistant\n" in decoded:
            return decoded.split("assistant\n")[-1].replace("<|im_end|>", "").strip()
        return decoded.strip()
    
    print("\n--- 💬 Interactive Chat Session Started ---")
    text_input = widgets.Textarea(placeholder='Enter English text here...', layout=widgets.Layout(width='70%', height='60px'))
    button = widgets.Button(description='Translate 🚀', button_style='success', layout=widgets.Layout(width='25%', height='60px'))
    output_area = widgets.Output()
    
    def on_button_clicked(b):
        with output_area:
            clear_output()
            eng = text_input.value.strip()
            if eng:
                print(f"⏳ Processing: {eng}...")
                res = generate_reply(eng)
                clear_output()
                print(f"📝 Input: {eng}\n")
                print(f"🤖 SVOMPTR Output:\n{res}\n")
            else:
                print("⚠️ Please enter some text.")
    
    button.on_click(on_button_clicked)
    display(widgets.HBox([text_input, button]))
    display(output_area)
else:
    print("❌ Final Output Error: Model is not loaded.")